In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session
import warnings
warnings.filterwarnings("ignore")

# House Prices Prediction: RandomForest Regression + RMSE

## Introduction
The Ames Housing dataset is a well-known regression dataset often used for Kaggle competitions.  
It contains detailed information on residential homes in Ames, Iowa, and the goal is to predict the final sale price of each house.  

In this notebook, we will:  
- Perform exploratory data analysis (EDA)  
- Preprocess missing values and categorical variables  
- Train a RandomForestRegressor model  
- Evaluate its performance using RMSE (Root Mean Squared Error)  

## Step 1: Load Dataset

We will start by loading the training and test datasets for the House Prices competition.  
We'll explore the data and check basic information about features.


In [ ]:
df_train = pd.read_csv("/kaggle/input/house-prices-advanced-regression-techniques/train.csv")
df_test = pd.read_csv("/kaggle/input/house-prices-advanced-regression-techniques/test.csv")


df_train.head()



## Step 2: Exploratory Data Analysis

We will explore the target variable (`SalePrice`) and basic statistics of numerical features.  
We will also look at missing values and correlations to guide feature engineering.
 


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# SalePrice distribution
plt.figure(figsize = (8,5))
sns.histplot(df_train["SalePrice"], kde = True, color = "skyblue")
plt.title("SalePrice Distribution")
plt.show()

# Log-transform to see normality
plt.figure(figsize = (8,5))
sns.histplot(np.log1p(df_train["SalePrice"]), kde = True, color = "salmon")
plt.title("Log-Transformed SalePrice Distribution")
plt.show()

# Correlation heatmap for numerical features
num_cols = df_train.select_dtypes(include = ["int64","float64"]).columns
plt.figure(figsize = (18,15))
sns.heatmap(df_train[num_cols].corr(), annot = True, fmt = ".2f", cmap = "coolwarm")
plt.title("Correlation Heatmap of Numeric Features")
plt.show()

# Missing values
missing = df_train.isnull().sum()
missing = missing[missing > 0].sort_values(ascending = False)
plt.figure(figsize = (10,6))
sns.barplot(x = missing.values, y = missing.index, palette = "magma")
plt.title("Missing Values Count by Feature")
plt.show()


## Step 3: Data Preprocessing

- Handle missing values: numeric → median, categorical → most frequent.  
- Convert categorical variables to numeric via one-hot encoding.  
- Align train and test sets to have the same columns.


In [ ]:
from sklearn.impute import SimpleImputer

# Separate target
y = df_train["SalePrice"]
X_train = df_train.drop("SalePrice", axis = 1)
X_test = df_test.copy()

# Identify numeric and categorical columns
num_cols = X_train.select_dtypes(include = ["int64","float64"]).columns
cat_cols = X_train.select_dtypes(include = ["object"]).columns

# Impute missing values
num_imputer = SimpleImputer(strategy = "median")
cat_imputer = SimpleImputer(strategy = "most_frequent")

X_train[num_cols] = num_imputer.fit_transform(X_train[num_cols])
X_test[num_cols] = num_imputer.transform(X_test[num_cols])

X_train[cat_cols] = cat_imputer.fit_transform(X_train[cat_cols])
X_test[cat_cols] = cat_imputer.transform(X_test[cat_cols])

# One-hot encoding
X_train_encoded = pd.get_dummies(X_train, drop_first = True)
X_test_encoded = pd.get_dummies(X_test, drop_first = True)

# Align columns
X_train_encoded, X_test_encoded = X_train_encoded.align(X_test_encoded, join = "left", axis = 1, fill_value = 0)

X = X_train_encoded


## Step 4: Feature Engineering

- Check skewed numeric features and apply log1p transformation if skewness > 0.75  
- Create simple interaction features (example: OverallQual * GrLivArea)  
- Ensure all features are numeric and ready for modeling


In [ ]:
from scipy.stats import skew

# Identify numeric features
numeric_feats = X.select_dtypes(include = ["int64","float64"]).columns

# Compute skewness
skewed_feats = X[numeric_feats].apply(lambda x: skew(x.dropna()))
skewed_feats = skewed_feats[abs(skewed_feats) > 0.75].index

# Log-transform skewed features
for feat in skewed_feats:
    X[feat] = np.log1p(X[feat])
    X_test_encoded[feat] = np.log1p(X_test_encoded[feat])

# Example interaction feature
if "OverallQual" in X.columns and "GrLivArea" in X.columns:
    X["Qual_GrLiv"] = X["OverallQual"] * X["GrLivArea"]
    X_test_encoded["Qual_GrLiv"] = X_test_encoded["OverallQual"] * X_test_encoded["GrLivArea"]

print("Feature matrix shape after engineering:", X.shape)


## Step 5: RandomForest Model Training & Evaluation

- Train `RandomForestRegressor` using 5-fold cross-validation  
- Compute RMSE to understand prediction errors


In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score

# Random Forest model
rf = RandomForestRegressor(n_estimators = 200, random_state = 42, n_jobs = -1)

# Cross-validation RMSE
cv_scores = cross_val_score(
    rf,
    X,
    y,
    cv = 5,
    scoring = "neg_root_mean_squared_error",
    n_jobs = -1
)

print("Cross-validation RMSE scores:", -cv_scores)
print("Average RMSE:", -np.mean(cv_scores))


## Step 6: Feature Importance Visualization

- Visualize top 20 important features according to Random Forest


In [ ]:
# Fit model on full data
rf.fit(X, y)

# Feature importances
importances = rf.feature_importances_
feat_df = pd.DataFrame({"Feature": X.columns, "Importance": importances})
feat_df = feat_df.sort_values(by = "Importance", ascending = False).head(20)

# Plot
plt.figure(figsize = (10,6))
sns.barplot(x = "Importance", y = "Feature", data = feat_df, palette = "viridis")
plt.title("Top 20 Feature Importances")
plt.tight_layout()
plt.show()
 

## Conclusion, Insights & Next Steps

### Conclusion
- Random Forest model achieved reasonable RMSE using the preprocessed features.
- Cross-validation ensured that the model generalizes well to unseen data.
- Feature importance analysis highlighted the most influential features for predicting `SalePrice`.

### Insights
- Features such as `OverallQual`, `GrLivArea`, and `GarageCars` are among the top predictors of house prices.
- Skewed numeric features were log-transformed to improve model stability.
- Interaction features like `OverallQual * GrLivArea` can capture complex relationships and improve predictions.

### Next Steps
- **Try other models**: Gradient Boosting, XGBoost, or LightGBM for potential performance improvement.
- **Hyperparameter tuning**: Optimize Random Forest or other models for better RMSE.
- **Feature engineering**: Explore polynomial or interaction features, and consider domain-specific features.
- **Outlier analysis**: Identify and treat extreme SalePrice values to enhance model robustness.
- **Deployment**: Use the trained model to make predictions on new house listings.
